# 02 — KILT scan, matched by Wikidata ID (Plan Step 2, part 1)

**Why:** the v5 scan matched pages by title and found only 133 of 299 subject pages.
Titles change (renames, redirects, spelling); Wikidata IDs do not.

**What changes from `HALT_RAG_v5_scan.ipynb`**
1. Match by Wikidata ID first (PopQA `s_uri` ↔ KILT `wikidata_info.wikidata_id`), title only as a backup.
2. Count duplicate records, to explain 6.3M records vs 5,903,530 pages.
3. Save how each page was matched (`qid` or `title`).

Same as before: one full pass, 10,000-page random background (Algorithm R), checkpoints to Drive every 200,000 records, so a Colab reset can resume.

**Output (data folder):** `kilt_cache_v6.pkl`, `kilt_scan_v6_report.json`, `halt_rag_audit_v6.csv`.
**Run this one on Colab** (high-RAM runtime): it streams all of KILT and takes several hours. Then copy the three output files into `data\` on the laptop.

In [ ]:
!pip install -q "datasets==3.6.0"
# ── Setup: works on a laptop (VS Code) and on Colab ───────────────────────
import os, sys, subprocess
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    # Colab: get the code from GitHub (push your latest commit first) and mount Drive
    REPO, BRANCH = "/content/HALT-RAG", "lift_signal"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/Shaithra2107/HALT-RAG", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    # laptop: this notebook sits in <repo>/notebooks, so the repo is one level up
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))

from halt_rag.config import load_config, data_path, data_dir
cfg = load_config(os.path.join(REPO, "config.yaml"))
print("running on:", "Colab" if ON_COLAB else "laptop", "| data folder:", data_dir(cfg))

## Load the 300 pilot questions and attach their Wikidata IDs

In [ ]:
import pandas as pd, numpy as np, pickle, json
from datasets import load_dataset
from halt_rag.data.popqa import wikidata_qid, parse_answers, normalise_title
from halt_rag.data.kilt_scan import Targets, ScanState, scan, report

audit = pd.read_csv(data_path(cfg, "audit_v5"))
print(f"audit rows: {len(audit)}")

if "s_uri" not in audit.columns:
    # attach s_uri from PopQA: by id if the audit kept it, else by question text
    pq = load_dataset(cfg["questions"]["dataset"], split="test").to_pandas()
    key = "id" if "id" in audit.columns else "question"
    audit = audit.merge(pq[[key, "s_uri"]].drop_duplicates(key), on=key, how="left")
audit["qid"] = audit["s_uri"].apply(wikidata_qid)
print(f"questions with a Wikidata ID: {audit['qid'].notna().sum()} / {len(audit)}")

targets = Targets.from_frame(audit)
print(f"targets: {len(targets.keys)} unique pages | {len(targets.by_qid)} with a Wikidata ID")

## The scan (resumes from the last checkpoint if one exists)

In [ ]:
from itertools import islice
from tqdm.auto import tqdm
col = cfg["collection"]
CKPT  = data_path(cfg, "kilt_ckpt_v6")
CACHE = data_path(cfg, "kilt_cache_v6")

ds = load_dataset(col["kilt_dataset"], split="full", streaming=True, trust_remote_code=True)
it = iter(ds)

state = None
if os.path.exists(CACHE):
    print("finished cache already exists — delete it to re-scan"); state = pickle.load(open(CACHE, "rb"))["state"]
else:
    if os.path.exists(CKPT):
        ck = pickle.load(open(CKPT, "rb")); state = ck["state"]
        try:
            it.load_state_dict(ck["iter_state"]); print(f"resumed at {state.scanned:,}")
        except Exception as e:
            print(f"iterator restore failed ({e}); fast-forwarding {state.scanned:,}")
            for _ in tqdm(islice(it, state.scanned), total=state.scanned): pass

    def save_ckpt(st):
        ck = {"state": st}
        try: ck["iter_state"] = it.state_dict()
        except Exception: pass
        pickle.dump(ck, open(CKPT, "wb"))
        print(f"  checkpoint {st.scanned:,} | found {len(st.kilt_pages)}/{len(targets.keys)} "
              f"| background {len(st.reservoir)}")

    state = scan(tqdm(it, desc="KILT"), targets,
                 reservoir_size=col["reservoir_pages"], min_para_chars=col["min_para_chars"],
                 max_para_per_background=col["max_para_per_background_page"], seed=cfg["seed"],
                 state=state, checkpoint_every=col["checkpoint_every"], on_checkpoint=save_ckpt)

    pickle.dump({"state": state,
                 # same keys the v5 signals notebook reads:
                 "kilt_pages": state.kilt_pages, "distractor_pool": state.reservoir,
                 "match_method": state.match_method, "scanned": state.scanned,
                 "n_eligible": state.n_eligible}, open(CACHE, "wb"))
    if os.path.exists(CKPT): os.remove(CKPT)
    print("saved", CACHE)

## Report — send these numbers to Dr. Kasuni

In [ ]:
rep = report(state, targets)
json.dump(rep, open(data_path(cfg, "scan_report_v6"), "w"), indent=2)
for k, v in rep.items(): print(f"{k:32s} {v:,}" if isinstance(v, int) else f"{k:32s} {v}")
print("\nv5 (title only) found 133 / 299")

missing = sorted(targets.keys - set(state.kilt_pages))
print(f"\nstill missing: {len(missing)}  e.g. {missing[:10]}")

## Recompute M2 and save `halt_rag_audit_v6.csv`
M2 = does a gold answer appear on the subject's own page? Then tag each question with one of the four kinds of "no" that this step can tell apart:
`page_missing` · `answer_not_on_page` · `found` (retrieval-based kinds come in Step 3).

In [ ]:
import re
def norm(t): return re.sub(r"\s+", " ", str(t).lower().strip())
def answer_on_page(answers, paras):
    text = norm(" ".join(paras))
    return any(re.search(r"(?<![\w])" + re.escape(norm(a)) + r"(?![\w])", text)
               for a in answers if norm(a))

status, method = [], []
for _, r in audit.iterrows():
    key = normalise_title(r["wiki_title"])
    paras = state.kilt_pages.get(key)
    if paras is None:
        status.append("page_missing"); method.append(None)
    else:
        status.append("found" if answer_on_page(parse_answers(r["answers_list"]), paras)
                      else "answer_not_on_page")
        method.append(state.match_method.get(key))
audit["m2_status_v6"], audit["match_method_v6"] = status, method
print(pd.crosstab(audit["m2_status_v6"], audit["band"], margins=True))
if "m2_status_v5" in audit.columns:
    print("\nv5 -> v6:"); print(pd.crosstab(audit["m2_status_v5"], audit["m2_status_v6"]))
out = data_path(cfg, "halt_rag_audit_v6.csv")
audit.to_csv(out, index=False); print("saved", out)